<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта: Система способов оплаты</h2>

----

### Вариант задания № 16


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Разработка системы различных способов оплаты на базе класса PaymentMethod. Система предусматривает онлайн-оплату, банковский перевод и оплату наличными.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) создайте явную реализации интерфейса и управление зависимостями 


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [4]:
// Интерфейс для отправки чека
public interface IReceiptable
{
    void SendReceipt(string email);
}

// Интерфейс для возврата средств
public interface IRefundable
{
    void Refund(decimal amount);
}

// Базовый класс
public class PaymentMethod
{
    private int _paymentMethodId;
    private string _methodName;
    private decimal _minAmount;
    private decimal _commission;

    // --- новые атрибуты ---
    private decimal _maxAmount;
    private bool _isActive;
    private DateTime _lastTransactionDate;

    public int PaymentMethodId
    {
        get { return _paymentMethodId; }
        set { _paymentMethodId = value; }
    }

    public string MethodName
    {
        get { return _methodName; }
        set { _methodName = value; }
    }

    public decimal MinAmount
    {
        get { return _minAmount; }
        set { _minAmount = value; }
    }

    public decimal Commission
    {
        get { return _commission; }
        set { _commission = value; }
    }

    public string Currency { get; set; }

    // --- новые свойства ---
    public decimal MaxAmount
    {
        get { return _maxAmount; }
        set { _maxAmount = value; }
    }

    public bool IsActive
    {
        get { return _isActive; }
        set { _isActive = value; }
    }

    public DateTime LastTransactionDate
    {
        get { return _lastTransactionDate; }
        set { _lastTransactionDate = value; }
    }

    public PaymentMethod(int paymentMethodId, string methodName,
        decimal minAmount, decimal commission, decimal maxAmount = 1000000m)
    {
        PaymentMethodId = paymentMethodId;
        MethodName = methodName;
        MinAmount = minAmount;
        Commission = commission;
        Currency = "RUB";
        MaxAmount = maxAmount;
        IsActive = true;
    }

    public bool CheckMinimumAmount(decimal amount)
    {
        return amount >= MinAmount;
    }

    // --- новый метод ---
    public bool CheckMaximumAmount(decimal amount)
    {
        return amount <= MaxAmount;
    }

    public decimal CalculateCommission(decimal amount)
    {
        return amount * (Commission / 100);
    }

    public void CancelPayment()
    {
        Console.WriteLine($"Платеж по способу '{MethodName}' отменен.");
    }

    // --- базовый метод ProcessPayment (для перекрытия) ---
    // ИСПРАВЛЕНО: теперь возвращает bool, чтобы наследники знали об успехе/неудаче
    // и не выполняли свою логику (списание/перевод), если платеж не прошел
    public virtual bool ProcessPayment(decimal amount)
    {
        if (!IsActive)
        {
            Console.WriteLine($"Способ оплаты '{MethodName}' временно неактивен.");
            return false;
        }

        if (!CheckMaximumAmount(amount))
        {
            Console.WriteLine(
                $"Ошибка: сумма {amount} {Currency} превышает максимальную ({MaxAmount} {Currency}).");
            return false;
        }

        if (!CheckMinimumAmount(amount))
        {
            Console.WriteLine(
                $"Ошибка: сумма {amount} {Currency} меньше минимальной " +
                $"({MinAmount} {Currency}) для '{MethodName}'.");
            return false;
        }

        decimal commission = CalculateCommission(amount);
        LastTransactionDate = DateTime.Now;

        Console.WriteLine(
            $"Успешно: платеж на сумму {amount} {Currency} " +
            $"(комиссия: {commission} {Currency}) проведен через '{MethodName}'.");

        return true;
    }

    // --- перегрузка метода ProcessPayment (принимает еще комментарий) ---
    public bool ProcessPayment(decimal amount, string comment)
    {
        bool result = ProcessPayment(amount);
        Console.WriteLine($"Комментарий к платежу: {comment}");
        return result;
    }

    public virtual void GetPaymentDetails()
    {
        Console.WriteLine(
            $"ID способа: {PaymentMethodId} | Название: {MethodName} | " +
            $"Мин. сумма: {MinAmount} {Currency} | Макс. сумма: {MaxAmount} {Currency} | " +
            $"Комиссия: {Commission}% | Активен: {IsActive}");
    }
}

// Онлайн-оплата
public class OnlinePayment : PaymentMethod, IReceiptable
{
    private string _paymentUrl;

    public string PaymentUrl
    {
        get { return _paymentUrl; }
        set { _paymentUrl = value; }
    }

    public OnlinePayment(int paymentMethodId, string methodName,
        decimal minAmount, decimal commission, string paymentUrl)
        : base(paymentMethodId, methodName, minAmount, commission)
    {
        PaymentUrl = paymentUrl;
    }

    public void SendReceipt(string email)
    {
        Console.WriteLine($"Электронный чек отправлен на почту: {email}");
    }

    // --- перекрытие метода GetPaymentDetails ---
    public override void GetPaymentDetails()
    {
        base.GetPaymentDetails();
        Console.WriteLine($"URL для оплаты: {PaymentUrl}");
    }
}

// Банковский перевод
public class BankTransfer : PaymentMethod
{
    private string _bankData;

    public string BankData
    {
        get { return _bankData; }
        set { _bankData = value; }
    }

    public BankTransfer(int paymentMethodId, string methodName,
        decimal minAmount, decimal commission, string bankData)
        : base(paymentMethodId, methodName, minAmount, commission)
    {
        BankData = bankData;
    }

    // --- перекрытие ProcessPayment ---
    // ИСПРАВЛЕНО: перевод отправляется только если базовая проверка прошла успешно
    public override bool ProcessPayment(decimal amount)
    {
        if (!base.ProcessPayment(amount))
            return false;

        Console.WriteLine($"Перевод отправлен по реквизитам: {BankData}");
        return true;
    }
}

// Оплата наличными
public class CashPayment : PaymentMethod
{
    private string _cashPickupPoint;

    public string CashPickupPoint
    {
        get { return _cashPickupPoint; }
        set { _cashPickupPoint = value; }
    }

    public CashPayment(int paymentMethodId, string methodName,
        decimal minAmount, decimal commission, string cashPickupPoint)
        : base(paymentMethodId, methodName, minAmount, commission)
    {
        CashPickupPoint = cashPickupPoint;
    }

    // --- перекрытие ProcessPayment ---
    // ИСПРАВЛЕНО: сообщение о точке выдачи только при успешной оплате
    public override bool ProcessPayment(decimal amount)
    {
        if (!base.ProcessPayment(amount))
            return false;

        Console.WriteLine($"Ожидайте оплату наличными в точке: {CashPickupPoint}");
        return true;
    }
}

// Оплата банковской картой
public class CardPayment : OnlinePayment, IRefundable
{
    public string CardNumber { get; set; }
    public string CardHolder { get; set; }

    public CardPayment(int paymentMethodId, string methodName,
        decimal minAmount, decimal commission, string paymentUrl,
        string cardNumber, string cardHolder)
        : base(paymentMethodId, methodName, minAmount, commission, paymentUrl)
    {
        CardNumber = cardNumber;
        CardHolder = cardHolder;
    }

    public void Refund(decimal amount)
    {
        Console.WriteLine(
            $"Оформлен возврат {amount} {Currency} на карту {CardNumber} ({CardHolder})");
    }

    // --- перегрузка метода Refund ---
    public void Refund(decimal amount, string reason)
    {
        Refund(amount);
        Console.WriteLine($"Причина возврата: {reason}");
    }

    // --- перекрытие ProcessPayment ---
    // ИСПРАВЛЕНО: списание с карты только при успешной оплате
    public override bool ProcessPayment(decimal amount)
    {
        if (!base.ProcessPayment(amount))
            return false;

        Console.WriteLine($"Списание с карты {CardNumber}, держатель: {CardHolder}");
        return true;
    }
}

// Generic класс для хранения и обработки коллекции способов оплаты
public class PaymentCollection<T> where T : PaymentMethod
{
    private List<T> _payments = new List<T>();

    public void Add(T payment)
    {
        _payments.Add(payment);
    }

    public void ProcessAll(decimal amount)
    {
        foreach (var payment in _payments)
        {
            payment.ProcessPayment(amount);
        }
    }

    public void ShowAllDetails()
    {
        foreach (var payment in _payments)
        {
            payment.GetPaymentDetails();
        }
    }
}

// Класс заказа
public class Order
{
    public int OrderId { get; set; }
    public decimal TotalAmount { get; set; }

    public Order(int orderId, decimal totalAmount)
    {
        OrderId = orderId;
        TotalAmount = totalAmount;
    }

    public void Pay(PaymentMethod paymentMethod)
    {
        Console.WriteLine(
            $"\nПопытка оплаты заказа №{OrderId} на сумму {TotalAmount} руб.");

        bool success = paymentMethod.ProcessPayment(TotalAmount);
        Console.WriteLine(success
            ? $"Заказ №{OrderId} успешно оплачен."
            : $"Заказ №{OrderId} НЕ оплачен.");
    }
}


// --- Создание объектов ---
BankTransfer bank = new BankTransfer(
    1, "Банковский перевод", 500, 1.5m,
    "ПАО Сбербанк, Р/С 407...");

CashPayment cash = new CashPayment(
    2, "Наличные", 50, 0m,
    "Касса №1 (ул. Мира, 10)");

CardPayment card = new CardPayment(
    3, "Банковская карта онлайн", 100, 2.0m,
    "https://securepay.ru",
    "2200-****-****-1234",
    "Иван Иванов");

// --- Вывод информации (перекрытие GetPaymentDetails) ---
Console.WriteLine("--- Информация о способах оплаты ---");
bank.GetPaymentDetails();
Console.WriteLine();
cash.GetPaymentDetails();
Console.WriteLine();
card.GetPaymentDetails();
Console.WriteLine();

// --- Проверка перегрузки методов ---
Console.WriteLine("--- Проверка перегрузки методов ---");
card.ProcessPayment(1500);                              // обычный вызов
card.ProcessPayment(1500, "Оплата за заказ №55");       // перегруженный вызов
card.Refund(500);                                       // обычный возврат
card.Refund(500, "Товар не подошел");                   // перегруженный возврат

// --- Работа Generic класса ---
Console.WriteLine("\n--- Работа Generic класса PaymentCollection ---");
PaymentCollection<PaymentMethod> collection = new PaymentCollection<PaymentMethod>();
collection.Add(bank);
collection.Add(cash);
collection.Add(card);

collection.ShowAllDetails();
Console.WriteLine();
collection.ProcessAll(600);

// --- Взаимодействие с классом Order ---
Console.WriteLine("\n--- Взаимодействие с классом Order ---");
Order order1 = new Order(101, 1000);
Order order2 = new Order(102, 30);

order1.Pay(card);
order2.Pay(card);

--- Информация о способах оплаты ---
ID способа: 1 | Название: Банковский перевод | Мин. сумма: 500 RUB | Макс. сумма: 1000000 RUB | Комиссия: 1.5% | Активен: True

ID способа: 2 | Название: Наличные | Мин. сумма: 50 RUB | Макс. сумма: 1000000 RUB | Комиссия: 0% | Активен: True

ID способа: 3 | Название: Банковская карта онлайн | Мин. сумма: 100 RUB | Макс. сумма: 1000000 RUB | Комиссия: 2.0% | Активен: True
URL для оплаты: https://securepay.ru

--- Проверка перегрузки методов ---
Успешно: платеж на сумму 1500 RUB (комиссия: 30.00 RUB) проведен через 'Банковская карта онлайн'.
Списание с карты 2200-****-****-1234, держатель: Иван Иванов
Успешно: платеж на сумму 1500 RUB (комиссия: 30.00 RUB) проведен через 'Банковская карта онлайн'.
Списание с карты 2200-****-****-1234, держатель: Иван Иванов
Комментарий к платежу: Оплата за заказ №55
Оформлен возврат 500 RUB на карту 2200-****-****-1234 (Иван Иванов)
Оформлен возврат 500 RUB на карту 2200-****-****-1234 (Иван Иванов)
Причина возврата: